In [50]:
import pandas as pd
import os
from openai import AsyncOpenAI
import asyncio
from tqdm import tqdm

In [51]:
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [52]:
data_real = pd.read_csv("../data/processed/train.csv")
data_synthetic = pd.read_csv("../data_synthetic/synthetic_raw_1_5k.csv")

In [53]:
data_real.head()

,label,text,label_numeric
0,negative,жизнь сука опасная!!!!!,0
1,positive,"В моей жизни есть пара человек, переписки с ко...",2
2,neutral,"Луна зевает на тропарь,\nКомета подметает лед,...",1
3,negative,"Знаете как обидно, когда последние деньки лета...",0
4,negative,"Трио закачайся : потерянная шлюха Кирилюк, ста...",0


In [54]:
data_synthetic.head()

,text,label
0,"на работе сегодня тестируют новую систему, пок...",neutral
1,Вчера сходили с друзьями в новый бар — атмосфе...,positive
2,"сегодня на кухне попробовал новый рецепт супа,...",neutral
3,"щас вернулся с концерта, голос леко сносит, но...",positive
4,"босс похвалил мою презентацию, настроение на о...",positive


In [58]:
data_synthetic.sample(200)['text'].to_list()

['запланировал на выходные уборку и мелкий ремонт, список покупок уже готов',
 'опять задержали зарплату, уже достало этот бардак на работе, не знаю что делать 😒',
 'на работе поменяли график, теперь начинаем в 9:30 вместо 9:00',
 'сегодня в метро все спокойно, поезда идут по расписанию, без задержек',
 'завтра тестирую новую кофемашину, посмотрю, как она себя ведёт.',
 'Опять задержка по зарплате, начальник молчит, устал уже от этого беспредела...',
 'На работе поменяли систему учёта, пока разбираюсь, вопросов много',
 'Опять опоздала на работу из‑за пробок, босс вешает на меня всю ответственность, сил нет 😒',
 'такая погодка и новый плейлист — ехать домой с улыбкой, кайфовый день 😊',
 'Утром по дороге на работу заметил новую кофейню, откроется на следующей неделе.',
 'В плейлисте случайно включил старый альбом, сейчас слушаю без пауз',
 'Наконец-то встретились с друзьями, пицца, много смеха и планов — идеальный вечер 😄',
 'Сегодня спонтанный пикник с друзьями, смеялись до слёз, заряд

In [ ]:
model = "gpt-5.4-mini-2026-03-17"

semaphore = asyncio.Semaphore(20)

decoding_params = {
    "temperature": 0.0,
    "max_completion_tokens": 5
}


PROMPT_TEMPLATE = """
You are an expert in detecting LLM-generated Russian social media text.

Task:
Determine whether the following VK-style comment/post is:

- real  → authentic human-written internet content
- synthetic → likely generated by an LLM

Analyze deeply.

Important:
Do NOT rely only on grammar quality.

Focus on distributional realism, cultural authenticity,
and human internet behavior patterns.

REAL Russian VK comments often contain:
- chaotic or broken formatting
- inconsistent punctuation
- random capitalization
- excessive brackets )))) or symbols
- internet slang from different eras
- emotional overreaction
- cringe or awkward phrasing
- copied statuses / repost-style text
- strange storytelling
- low coherence
- abrupt emotional or topic shifts
- niche references, names, local context
- emotionally unstable writing
- weird toxicity
- excessive length or extremely short messages
- accidental grammar mistakes
- unusual sentence rhythm
- authentic human randomness

REAL comments may look:
- messy
- uncomfortable
- overly personal
- irrational
- emotionally intense
- culturally specific
- outdated
- bizarre

SYNTHETIC comments often:
- sound balanced and controlled
- are too readable
- are structurally repetitive
- use safe emotional patterns
- avoid genuine weirdness
- contain generic daily-life scenarios
- overuse templates like:
  "today..."
  "again..."
  "on my way to work..."
  "playlist..."
  "coffee..."
  "boss..."
  "mood..."
- maintain consistent tone
- feel optimized for clarity
- lack authentic social-network chaos
- have artificially clean emotional arcs
- resemble short assistant-generated diary entries

Strong synthetic indicators:
- repetitive structure across samples
- similar sentence rhythm
- polished neutrality
- predictable emotions
- generic urban situations
- low cultural entropy
- “LLM smoothness”

Examples:

REAL:
"сломалсяпробелнаклавиатуре"

REAL:
"Если первым не писать людям и не навязываться, то можно обнаружить, что, в принципе, никому ты нахрен и не нужен."

REAL:
"Ты там не сдох еще?)"

SYNTHETIC:
"сегодня в метро все спокойно, поезда идут по расписанию, без задержек"

SYNTHETIC:
"завтра тестирую новую кофемашину, посмотрю, как она себя ведёт."

SYNTHETIC:
"сегодня на работе дали новую задачу, завтра разберусь, выглядит нормально"

Text:
----------------
{txt}
----------------

Think carefully.

Return ONLY one word:
real
or
synthetic
"""

VALID_LABELS = {"real", "synthetic"}


def normalize_label(t):
    t = t.strip().lower()
    return t if t in VALID_LABELS else "synthetic"


async def classify_text(text):
    prompt = PROMPT_TEMPLATE.format(txt=text)

    try:
        async with semaphore:
            response = await client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                **decoding_params
            )

        raw = response.choices[0].message.content
        return normalize_label(raw)

    except Exception:
        return "synthetic"


async def process_all(texts, batch_size=50):
    results = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        tasks = [classify_text(t) for t in batch_texts]

        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)

    return results


# -----------------------------
# prepare balanced dataset
# -----------------------------

n = min(len(data_real), len(data_synthetic), 500)

real_df = data_real.sample(n=n, random_state=42)[["text"]].copy()
real_df["source"] = "real"

synthetic_df = data_synthetic.sample(n=n, random_state=42)[["text"]].copy()
synthetic_df["source"] = "synthetic"

df = pd.concat([real_df, synthetic_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

texts = df["text"].astype(str).tolist()

In [60]:
# -----------------------------
# inference
# -----------------------------

preds = await process_all(texts)

df["prediction"] = preds

100%|██████████| 20/20 [02:57<00:00,  8.87s/it]


In [62]:
# -----------------------------
# metrics
# -----------------------------

accuracy = (df["source"] == df["prediction"]).mean()

print(f"Accuracy: {accuracy:.4f}")

print("\nConfusion matrix:")
print(pd.crosstab(df["source"], df["prediction"]))

Accuracy: 0.7440

Confusion matrix:
prediction  real  synthetic
source                     
real         255        245
synthetic     11        489


In [61]:
# Accuracy: 0.6830

# Confusion matrix:
# prediction  real  synthetic
# source                     
# real         565        435
# synthetic    199        801

In [63]:

# -----------------------------
# inspect mistakes
# -----------------------------

fp = df[
    (df["source"] == "real") &
    (df["prediction"] == "synthetic")
]

fn = df[
    (df["source"] == "synthetic") &
    (df["prediction"] == "real")
]

print("\nReal -> Synthetic mistakes:")
display(fp.head(10))

print("\nSynthetic -> Real mistakes:")
display(fn.head(10))


Real -> Synthetic mistakes:


,text,source,prediction
11,"За тысячи километров от родных рук и глаз,ты н...",real,synthetic
19,"Ну, что? Ваша весна наступила. Дождались? И че...",real,synthetic
21,День Победы\n,real,synthetic
22,"Очень по тебе скучаю, братишка :(",real,synthetic
33,Моя слабость...,real,synthetic
34,"Анна, я подобрал тебе шапку, чтобы голова не ...",real,synthetic
39,В один прекрасный день я\nне зайду на сайт.Одн...,real,synthetic
41,"уж очень много, но к сажелению, прошлое не вер...",real,synthetic
45,О девонька моя! Моя принцесса!\nТы заняла во м...,real,synthetic
52,вот как то так.,real,synthetic



Synthetic -> Real mistakes:


,text,source,prediction
86,"подруга подсидела меня перед всей компанией, р...",synthetic,real
148,"опять задержали зарплату, начальник только обе...",synthetic,real
163,"опять задержали зп, начальник врет, хочу уволи...",synthetic,real
279,"вечно тормозит связь когда фильм на пике, заст...",synthetic,real
385,"Опять задержали зарплату, начальник только отм...",synthetic,real
591,"опять таскают ночные смены без доплаты, устал ...",synthetic,real
593,опять начальник залетает с правками в 11 вечер...,synthetic,real
643,снова пропало интернет в самый ответственный м...,synthetic,real
745,"опять задержали зарплату, начальство отмазки д...",synthetic,real
803,"опять начальник залез с правками в 11 вечера, ...",synthetic,real


In [65]:
# REAL -> SYNTHETIC
real_as_synth = df[
    (df["source"] == "real") &
    (df["prediction"] == "synthetic")
]

# SYNTHETIC -> REAL
synth_as_real = df[
    (df["source"] == "synthetic") &
    (df["prediction"] == "real")
]

print("REAL COMMENTS PREDICTED AS SYNTHETIC")

for txt in real_as_synth["text"].sample(
    min(50, len(real_as_synth)),
    random_state=42
):
    print(txt)

print("\n\n")
print("SYNTHETIC COMMENTS PREDICTED AS REAL")

for txt in synth_as_real["text"].sample(
    min(50, len(synth_as_real)),
    random_state=42
):
    print(txt)

REAL COMMENTS PREDICTED AS SYNTHETIC
Мы сделали это!!!
В один прекрасный день я
не зайду на сайт.Одни
подумают,что я занят,другие, что на счету нет,многие и вовсе не заметят
моего отсутствия,а в это
время Братья,с
осторожностью,будут
укладывать мое тело в
могилу
"ОНА ПРЕВРАТИЛАСЬ В ОДНУ-ЕДИНСТВЕННУЮ ОГРОМНУЮ МУРАШКУ И СКАЗАЛА "ДА"."
Друзья, имею что спросить за ваше мнение... Рождается песня мощная... Опускаю все подробности... Даю только припев (да и то не весь) (кто поймет - тот поймет сразу)... "Отражение искаженной нереальности в кривом зеркале" (с). И тут вдруг, чего раньше не бывало, дилемма: политизировать текст или нет? Движется параллельно, а надо остановиться и сконцентрироваться на одном варианте... Жду отзывов...
Прекрасная пара Максим и Мария... Она сказала:"ДА!!!". Было очень приятно стать частичкой вашей истории любви!
Нельзя так пить...! Особенно посреди рабочей недели...
Не забывай меня!!!
Роман Манекин - ПАМЯТИ ПАВШИХ БУДЕМ ДОСТОЙНЫ?

Так.... | Facebook
вот как то так

In [66]:
# check decoding params

data_synthetic_decoding_params_tunning = pd.read_csv("../data_synthetic/synthetic_decoding_params_1_5k.csv")

n = 500

synthetic_df = data_synthetic_decoding_params_tunning.sample(
    n=n,
    random_state=42
)[["text"]].copy()

texts = synthetic_df["text"].astype(str).tolist()

preds = await process_all(texts)

synthetic_df["prediction"] = preds


synthetic_rate = (
    (synthetic_df["prediction"] == "synthetic")
    .mean()
)

real_rate = (
    (synthetic_df["prediction"] == "real")
    .mean()
)

print(f"Predicted synthetic: {synthetic_rate:.4f}")
print(f"Predicted real: {real_rate:.4f}")

100%|██████████| 10/10 [01:05<00:00,  6.55s/it]

Predicted synthetic: 0.5980
Predicted real: 0.4020


In [68]:
# synthetic -> synthetic
synthetic_as_synthetic = synthetic_df[
    synthetic_df["prediction"] == "synthetic"
].sample(50, random_state=42)

print("=" * 80)
print("SYNTHETIC → SYNTHETIC")
print("=" * 80)

for i, row in enumerate(synthetic_as_synthetic.itertuples(), 1):
    print(row.text)


# synthetic -> real
synthetic_as_real = synthetic_df[
    synthetic_df["prediction"] == "real"
].sample(50, random_state=42)

print("\n" + "=" * 80)
print("SYNTHETIC → REAL")
print("=" * 80)

for i, row in enumerate(synthetic_as_real.itertuples(), 1):
    print(row.text)

SYNTHETIC → SYNTHETIC
ааа вот это кайф!!! прям красота, аж улыбнуло 😍🔥
Да вчера тоже увидел, сначала вообще не понял что за тема... потом вник и всё поехало как-то само
в ленте опять это вылезло, пролистнул и все
ооо дааа, вот это прям греет душу 😍🔥
внатуре странно всё это, просто посмотрел и остался с ощущением какого‑то непонятного фона...
под вечер опять все куда-то срочно пропали, я только чай налил и с ним встрял... короче ладно
ааа вот это кайф щас прям прочитал и аж улыбнуло 😄🔥
ооо вот это кайф 😍🔥 прям захотелось ещё посмотреть!!
Да забавно вышло, просто как-то на бегу заметил и всё
ооо вот это пушечкааа!!!🔥😍 прям зашло жоско, пересмотрел два раза и теперь хожу довольный как слон, кайфищщщеее...
В автобусе опять белки бешеные... люди почему-то решили, что утро начинается с толкотни и дыхания в затылок 😑
Ооо, вот это прям огонь получился!!!👌🔥 аж настроение подпрыгнуло, кайфанул
вот это они накрутили с дизайном, глаза сначала поплыли... потом привык уже
Да я просто мимо увидел, ща